# Alpha Evaluator — Full Pipeline

**Two portfolio modes:**
1. **Simple Rank Portfolio** — fast, rank-based market-neutral (cell 4)
2. **Conviction Portfolio** — full GAM-style pipeline using pre-computed raw alphas from `raw_alphas.csv` (cells 5+)


In [1]:
import os, sys
import pandas as pd
import numpy as np
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = "/Users/ritikraj/Utkarsh:QRT_comp/QRT_comp"
sys.path.append(os.path.join(BASE_DIR, "phase2_qrt_challenge"))

from scripts.alpha_evaluator import (
    evaluate_single_alpha, 
    build_conviction_portfolio,
    evaluate_conviction_portfolio,
    Alpha101
)

RAW_CSV_PATH = os.path.join(BASE_DIR, "stores", "raw_alphas.csv")
README_PATH = os.path.join(BASE_DIR, "phase2_qrt_challenge", "alphas_performance.md")


In [2]:
# Load data
print("Loading data...")
returns = pd.read_parquet(os.path.join(BASE_DIR, "stores", "returns.parquet"))
universe = pd.read_parquet(os.path.join(BASE_DIR, "stores", "universe_5m.parquet"))
yf_data = pd.read_pickle(os.path.join(BASE_DIR, "top_5000_yf_data.pkl"))

adj_close = yf_data.xs('Close', level='Price', axis=1).loc[:, lambda df: ~df.columns.duplicated()]
volume = yf_data.xs('Volume', level='Price', axis=1).loc[:, lambda df: ~df.columns.duplicated()]

print(f"Data loaded — {len(returns)} days, {len(returns.columns)} tickers")

# Load pre-computed raw alphas from CSV
print(f"Loading raw alphas from {RAW_CSV_PATH}...")
raw_alphas_all = pd.read_csv(RAW_CSV_PATH, header=[0, 1], index_col=0, parse_dates=True)
available = raw_alphas_all.columns.get_level_values(0).unique().tolist()
print(f"Available pre-computed alphas: {available}")


Loading data...
Data loaded — 4111 days, 4997 tickers
Loading raw alphas from /Users/ritikraj/Utkarsh:QRT_comp/QRT_comp/stores/raw_alphas.csv...
Available pre-computed alphas: ['alpha_001', 'alpha_002', 'alpha_003', 'alpha_004', 'alpha_005', 'alpha_006', 'alpha_007', 'alpha_008', 'alpha_009', 'alpha_010', 'alpha_011', 'alpha_012', 'alpha_013', 'alpha_014', 'alpha_015', 'alpha_016', 'alpha_017', 'alpha_018', 'alpha_019', 'alpha_020', 'alpha_021', 'alpha_022', 'alpha_023', 'alpha_024', 'alpha_025']


---
## Mode 1: Compute & Save a New Alpha (Simple Rank Portfolio)
Use this to compute a new alpha that isn't in `raw_alphas.csv` yet.

In [3]:
alpha_number = 11

evaluate_single_alpha(
    alpha_num=alpha_number,
    yf_data_df=yf_data,
    returns_df=returns,
    universe_df=universe,
    raw_csv_path=RAW_CSV_PATH,
    readme_path=README_PATH
)


NameError: name 'yf_data' is not defined

---
## Mode 2: Conviction Portfolio (from pre-computed alphas)

Uses the **already saved** raw alpha from `raw_alphas.csv` — no recomputation needed.

Pipeline: ADV filter → Beta neutralization → Inverse-vol weighting → Z-score conviction + hysteresis → Position limits → Dollar-neutral L/S


In [3]:
# ══════════════════════════════════════════════════════════
# CONFIGURE PARAMETERS HERE
# ══════════════════════════════════════════════════════════

ALPHA_NUMBER = 11          # Must already exist in raw_alphas.csv

ENTRY_THRESHOLD = 2      # Z-score to enter a new position
EXIT_THRESHOLD = 0.5       # Z-score to hold an existing position
MIN_ADV_USD = 5_000_000    # Minimum 60-day ADV in USD
TARGET_GMV = 10_000_000    # Target gross market value
VOL_WINDOW = 20            # Realized vol lookback (days)
BETA_WINDOW = 250          # Rolling beta lookback (days)


In [4]:
# Load pre-computed alpha from CSV (no recomputation!)
alpha_name = f"alpha_{ALPHA_NUMBER:03d}"
assert alpha_name in available, f"{alpha_name} not found in raw_alphas.csv. Run Mode 1 first to compute it."

raw_alpha = raw_alphas_all[alpha_name]  # Extract the single alpha's Date x Ticker DataFrame
print(f"Loaded {alpha_name} from CSV — shape: {raw_alpha.shape}")


Loaded alpha_011 from CSV — shape: (4111, 4997)


In [5]:
from scripts.alpha_evaluator import build_newconviction_portfolio

portfolio = build_newconviction_portfolio(
    raw_alpha=raw_alpha.shift(1),  # Always keep the T+2 shift for Alpha 11!
    returns_df=returns,
    universe_df=universe,
    adj_close=adj_close,
    volume=volume,
    entry_threshold=0.0,
    exit_threshold=0.0,
    neutralize_beta=False,
    use_vol_weighting=False,
    rank_signal=True,
    
    # --- REGIME FILTERS TO PLAY WITH ---
    flag_parabolic=True,         # BANS shorting stocks >30% above 20-day average
    flag_market_breadth=False,   # CUTS portfolio by 75% if entire market is melting up
    flag_squeeze_vol=False       # BANS shorting stocks where volatility spikes while going up
)


    Computing regime filter metrics...
    Building new conviction portfolio day-by-day...


In [6]:
# Evaluate YoY performance
print(f"\nEvaluating {alpha_name} conviction portfolio...\n")
metrics, overall = evaluate_conviction_portfolio(
    portfolio=portfolio,
    returns_df=returns,
    raw_alpha=raw_alpha,
    label=alpha_name
)

print("=" * 65)
print(f"  {alpha_name.upper()} — CONVICTION (z_entry={ENTRY_THRESHOLD}, z_exit={EXIT_THRESHOLD})")
print("=" * 65)
print(f"  Overall Net Sharpe:   {overall['Net Sharpe']}")
print(f"  Overall Gross Sharpe: {overall['Gross Sharpe']}")
print(f"  Overall Turnover:     {overall['Turnover']}")
print(f"  Overall Mean IC:      {overall['IC']}")
print("-" * 65)

df_yoy = pd.DataFrame(metrics)
if 'Year' in df_yoy.columns:
    df_yoy = df_yoy.set_index('Year')
display(df_yoy)



Evaluating alpha_011 conviction portfolio...

  ALPHA_011 — CONVICTION (z_entry=2, z_exit=0.5)
  Overall Net Sharpe:   -0.606
  Overall Gross Sharpe: 0.808
  Overall Turnover:     101.227%
  Overall Mean IC:      0.0169
-----------------------------------------------------------------


,Net Sharpe,Gross Sharpe,Turnover,IC
Year,,,,
2010,-0.914,0.833,103.68%,0.0248
2011,0.033,1.857,102.298%,0.0168
2012,-0.539,1.366,101.28%,0.0243
2013,-0.209,1.575,100.635%,0.0161
2014,-1.893,-0.031,100.172%,0.0183
2015,-0.257,1.192,99.146%,0.0245
2016,-2.531,-1.097,100.623%,0.0132
2017,-1.261,0.998,99.843%,0.0153
2018,-2.267,-0.307,100.059%,0.0105


---
## View All Saved Performance Results

In [ ]:
if os.path.exists(README_PATH):
    with open(README_PATH, "r") as f:
        display(Markdown(f.read()))
else:
    print("No performance file found yet.")


## Alpha 001


**Overall** — Net Sharpe: `-1.016` · Gross Sharpe: `-0.517` · Turnover: `91.226%` · Mean IC: `-0.0139`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.449 | -1.453 | 92.484% | -0.022 |
| 2011 | -1.573 | -0.847 | 90.632% | -0.0134 |
| 2012 | -3.99 | -2.837 | 91.21% | -0.0151 |
| 2013 | -0.126 | 0.978 | 89.883% | -0.0049 |
| 2014 | -1.632 | -0.783 | 91.021% | -0.012 |
| 2015 | -1.68 | -0.935 | 92.252% | -0.0156 |
| 2016 | -2.399 | -1.818 | 90.556% | -0.0167 |
| 2017 | -0.731 | 0.25 | 90.226% | -0.0106 |
| 2018 | -1.098 | -0.196 | 90.873% | -0.0094 |
| 2019 | -2.112 | -1.327 | 90.382% | -0.009 |
| 2020 | -0.932 | -0.663 | 90.532% | -0.0189 |
| 2021 | -0.017 | 0.382 | 90.669% | -0.0085 |
| 2022 | -1.03 | -0.689 | 90.286% | -0.0186 |
| 2023 | -2.478 | -1.814 | 90.611% | -0.0154 |
| 2024 | -0.451 | 0.123 | 91.132% | -0.0124 |
| 2025 | 0.077 | 0.33 | 91.693% | -0.0182 |
| 2026 | -1.901 | -1.455 | 91.113% | -0.0185 |

## Alpha 002


**Overall** — Net Sharpe: `-0.243` · Gross Sharpe: `0.328` · Turnover: `55.582%` · Mean IC: `0.0069`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.241 | 0.758 | 55.322% | 0.0092 |
| 2011 | 1.632 | 2.504 | 55.304% | 0.0081 |
| 2012 | 0.441 | 1.592 | 55.137% | 0.0085 |
| 2013 | 0.157 | 1.33 | 55.468% | 0.0079 |
| 2014 | -0.474 | 0.435 | 55.355% | 0.0074 |
| 2015 | -0.308 | 0.554 | 55.107% | 0.0065 |
| 2016 | -2.162 | -1.373 | 55.444% | 0.0041 |
| 2017 | 0.138 | 1.247 | 55.698% | 0.0067 |
| 2018 | 0.647 | 1.634 | 55.523% | 0.0065 |
| 2019 | -1.3 | -0.384 | 55.394% | 0.0058 |
| 2020 | -0.696 | -0.318 | 55.764% | 0.007 |
| 2021 | -0.345 | 0.227 | 55.694% | 0.0089 |
| 2022 | -0.201 | 0.364 | 55.026% | 0.004 |
| 2023 | -0.946 | -0.127 | 55.103% | 0.0036 |
| 2024 | -0.682 | -0.005 | 55.144% | 0.0072 |
| 2025 | 0.451 | 0.665 | 55.527% | 0.009 |
| 2026 | -3.593 | -2.949 | 54.713% | 0.0058 |

## Alpha 003


**Overall** — Net Sharpe: `-0.465` · Gross Sharpe: `-0.011` · Turnover: `44.545%` · Mean IC: `0.0045`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.525 | 1.329 | 45.017% | 0.0044 |
| 2011 | 1.021 | 1.786 | 44.874% | 0.0044 |
| 2012 | 2.022 | 2.886 | 45.059% | 0.0042 |
| 2013 | -0.334 | 0.504 | 44.683% | 0.004 |
| 2014 | -0.915 | -0.2 | 44.696% | 0.0028 |
| 2015 | -0.821 | -0.098 | 44.433% | 0.0035 |
| 2016 | 0.549 | 1.197 | 44.682% | 0.004 |
| 2017 | -1.474 | -0.671 | 44.698% | 0.0005 |
| 2018 | 1.263 | 2.105 | 44.656% | 0.0054 |
| 2019 | 1.2 | 1.96 | 44.365% | 0.0075 |
| 2020 | -0.912 | -0.7 | 43.683% | 0.0049 |
| 2021 | -0.944 | -0.608 | 43.934% | 0.0008 |
| 2022 | -1.036 | -0.699 | 43.757% | 0.0016 |
| 2023 | -1.735 | -1.181 | 44.291% | 0.0065 |
| 2024 | -3.142 | -2.704 | 44.03% | 0.0058 |
| 2025 | 0.346 | 0.636 | 43.767% | 0.01 |
| 2026 | -1.073 | -0.765 | 43.104% | 0.0063 |

## Alpha 004


**Overall** — Net Sharpe: `0.196` · Gross Sharpe: `0.471` · Turnover: `65.923%` · Mean IC: `0.014`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.639 | 1.101 | 67.257% | 0.02 |
| 2011 | 0.383 | 0.712 | 66.491% | 0.0212 |
| 2012 | -0.215 | 0.394 | 66.957% | 0.017 |
| 2013 | -0.269 | 0.459 | 66.709% | 0.0133 |
| 2014 | 0.091 | 0.558 | 66.252% | 0.0138 |
| 2015 | -0.608 | -0.214 | 65.966% | 0.0142 |
| 2016 | -0.479 | -0.151 | 66.249% | 0.0108 |
| 2017 | -0.022 | 0.601 | 66.099% | 0.0133 |
| 2018 | 0.543 | 0.95 | 66.186% | 0.0112 |
| 2019 | -0.15 | 0.28 | 65.85% | 0.0107 |
| 2020 | 0.711 | 0.836 | 64.487% | 0.0245 |
| 2021 | -0.426 | -0.233 | 64.43% | 0.0108 |
| 2022 | 0.455 | 0.62 | 64.627% | 0.0073 |
| 2023 | 0.656 | 1.013 | 64.479% | 0.0105 |
| 2024 | 0.16 | 0.545 | 64.294% | 0.0132 |
| 2025 | 1.045 | 1.273 | 64.295% | 0.0128 |
| 2026 | -0.636 | -0.425 | 64.996% | 0.0118 |

## Alpha 005


**Overall** — Net Sharpe: `0.188` · Gross Sharpe: `0.595` · Turnover: `79.458%` · Mean IC: `0.0285`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.888 | 1.608 | 81.254% | 0.0399 |
| 2011 | 0.731 | 1.222 | 80.338% | 0.0339 |
| 2012 | 0.157 | 0.939 | 79.957% | 0.0329 |
| 2013 | -0.856 | 0.026 | 80.323% | 0.0278 |
| 2014 | -0.356 | 0.165 | 79.337% | 0.0239 |
| 2015 | 0.221 | 0.715 | 78.444% | 0.0309 |
| 2016 | 0.436 | 0.904 | 78.695% | 0.0279 |
| 2017 | -0.081 | 0.64 | 78.851% | 0.0261 |
| 2018 | 0.129 | 0.733 | 76.641% | 0.0276 |
| 2019 | 0.654 | 1.219 | 80.367% | 0.0268 |
| 2020 | 1.802 | 2.025 | 82.052% | 0.0416 |
| 2021 | -0.522 | -0.285 | 78.043% | 0.0192 |
| 2022 | -0.2 | 0.054 | 76.49% | 0.0201 |
| 2023 | 0.956 | 1.485 | 78.889% | 0.0245 |
| 2024 | -0.507 | -0.005 | 79.088% | 0.026 |
| 2025 | -1.037 | -0.625 | 78.375% | 0.0274 |
| 2026 | -0.059 | 0.263 | 76.711% | 0.0277 |

## Alpha 006


**Overall** — Net Sharpe: `-0.361` · Gross Sharpe: `-0.019` · Turnover: `44.767%` · Mean IC: `0.0056`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.484 | 1.14 | 45.624% | 0.0036 |
| 2011 | 1.174 | 1.711 | 44.722% | 0.007 |
| 2012 | 0.698 | 1.472 | 45.422% | 0.0052 |
| 2013 | -0.455 | 0.376 | 45.775% | 0.0036 |
| 2014 | -0.634 | -0.135 | 44.659% | 0.0002 |
| 2015 | -0.559 | -0.083 | 44.305% | 0.0059 |
| 2016 | -0.255 | 0.205 | 44.994% | 0.0022 |
| 2017 | -0.795 | -0.162 | 45.257% | 0.0032 |
| 2018 | 1.063 | 1.66 | 44.872% | 0.0074 |
| 2019 | -0.197 | 0.401 | 44.84% | 0.0066 |
| 2020 | -0.125 | 0.064 | 44.079% | 0.0084 |
| 2021 | -0.111 | 0.121 | 43.701% | 0.0073 |
| 2022 | -0.154 | 0.05 | 43.525% | 0.0082 |
| 2023 | -1.155 | -0.713 | 44.523% | 0.0066 |
| 2024 | -2.5 | -2.113 | 44.341% | 0.0052 |
| 2025 | -1.04 | -0.843 | 44.023% | 0.0084 |
| 2026 | -1.298 | -1.029 | 43.751% | 0.0023 |

## Alpha 007


**Overall** — Net Sharpe: `-0.393` · Gross Sharpe: `0.027` · Turnover: `103.366%` · Mean IC: `0.0141`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.107 | 0.603 | 106.445% | 0.0138 |
| 2011 | 0.827 | 1.707 | 104.631% | 0.0165 |
| 2012 | -0.776 | 0.34 | 104.267% | 0.0163 |
| 2013 | -2.314 | -1.338 | 102.952% | 0.0117 |
| 2014 | -0.78 | -0.032 | 101.382% | 0.0135 |
| 2015 | 0.509 | 1.214 | 104.146% | 0.0179 |
| 2016 | 0.447 | 1.027 | 103.159% | 0.0146 |
| 2017 | -0.363 | 0.449 | 104.63% | 0.0123 |
| 2018 | 1.151 | 1.922 | 99.327% | 0.0142 |
| 2019 | 0.293 | 1.132 | 103.776% | 0.0135 |
| 2020 | -0.328 | -0.119 | 101.87% | 0.0204 |
| 2021 | -0.624 | -0.283 | 104.151% | 0.0132 |
| 2022 | -0.574 | -0.252 | 101.922% | 0.0089 |
| 2023 | -0.374 | 0.197 | 100.742% | 0.0123 |
| 2024 | -1.757 | -1.552 | 102.096% | 0.0119 |
| 2025 | -0.169 | 0.238 | 102.026% | 0.0157 |
| 2026 | 0.064 | 0.445 | 98.906% | 0.01 |

## Alpha 008


**Overall** — Net Sharpe: `0.089` · Gross Sharpe: `0.384` · Turnover: `54.129%` · Mean IC: `0.016`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.542 | 1.07 | 56.095% | 0.0194 |
| 2011 | 1.162 | 1.556 | 54.468% | 0.0258 |
| 2012 | 0.137 | 0.731 | 54.607% | 0.0215 |
| 2013 | 1.01 | 1.679 | 54.357% | 0.0176 |
| 2014 | -0.851 | -0.409 | 52.981% | 0.0113 |
| 2015 | 0.082 | 0.471 | 54.277% | 0.0252 |
| 2016 | -1.187 | -0.852 | 52.689% | 0.006 |
| 2017 | -0.089 | 0.46 | 53.675% | 0.0174 |
| 2018 | 0.431 | 0.819 | 52.462% | 0.0132 |
| 2019 | 0.067 | 0.482 | 54.412% | 0.0113 |
| 2020 | 0.981 | 1.134 | 56.515% | 0.0231 |
| 2021 | -0.269 | -0.072 | 53.122% | 0.0165 |
| 2022 | 0.24 | 0.427 | 54.165% | 0.0132 |
| 2023 | 0.168 | 0.559 | 54.08% | 0.0141 |
| 2024 | -0.295 | 0.034 | 51.872% | 0.0158 |
| 2025 | 0.107 | 0.386 | 53.149% | 0.0112 |
| 2026 | -2.027 | -1.823 | 52.148% | -0.0 |

## Alpha 009


**Overall** — Net Sharpe: `-0.618` · Gross Sharpe: `0.093` · Turnover: `126.859%` · Mean IC: `0.0252`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -1.285 | -0.137 | 127.504% | 0.0274 |
| 2011 | -1.28 | -0.308 | 127.54% | 0.0226 |
| 2012 | -2.069 | -0.735 | 126.439% | 0.0213 |
| 2013 | -2.896 | -1.441 | 126.462% | 0.0259 |
| 2014 | -1.542 | -0.571 | 126.398% | 0.0196 |
| 2015 | -0.329 | 0.711 | 127.968% | 0.032 |
| 2016 | 0.985 | 1.783 | 128.85% | 0.0394 |
| 2017 | 0.448 | 1.653 | 127.729% | 0.036 |
| 2018 | -0.601 | 0.479 | 125.288% | 0.0272 |
| 2019 | -1.049 | -0.081 | 125.889% | 0.0206 |
| 2020 | -1.386 | -0.972 | 126.767% | 0.0226 |
| 2021 | 0.726 | 1.258 | 125.712% | 0.0283 |
| 2022 | -2.097 | -1.658 | 124.348% | 0.0092 |
| 2023 | 1.294 | 2.158 | 125.516% | 0.0238 |
| 2024 | -1.447 | -0.696 | 124.68% | 0.0231 |
| 2025 | 0.422 | 0.871 | 125.762% | 0.025 |
| 2026 | -0.184 | 0.387 | 122.67% | 0.0216 |

## Alpha 010


**Overall** — Net Sharpe: `-0.955` · Gross Sharpe: `-0.197` · Turnover: `126.829%` · Mean IC: `0.0221`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.121 | -0.914 | 127.553% | 0.0224 |
| 2011 | -1.533 | -0.475 | 127.466% | 0.0197 |
| 2012 | -3.514 | -2.07 | 126.237% | 0.0167 |
| 2013 | -3.339 | -1.777 | 126.644% | 0.0237 |
| 2014 | -2.375 | -1.313 | 126.491% | 0.0156 |
| 2015 | -0.388 | 0.757 | 127.798% | 0.0307 |
| 2016 | 0.866 | 1.76 | 128.779% | 0.0374 |
| 2017 | -0.244 | 1.022 | 128.036% | 0.0325 |
| 2018 | -0.561 | 0.541 | 125.161% | 0.026 |
| 2019 | -1.795 | -0.706 | 125.742% | 0.02 |
| 2020 | -1.828 | -1.39 | 126.674% | 0.0174 |
| 2021 | 0.7 | 1.309 | 125.593% | 0.0247 |
| 2022 | -2.326 | -1.873 | 124.251% | 0.0043 |
| 2023 | 0.743 | 1.672 | 125.354% | 0.0207 |
| 2024 | -1.637 | -0.833 | 124.596% | 0.0216 |
| 2025 | 0.352 | 0.815 | 125.838% | 0.0206 |
| 2026 | -0.542 | 0.058 | 122.565% | 0.0195 |

## Alpha 011


**Overall** — Net Sharpe: `0.818` · Gross Sharpe: `1.506` · Turnover: `99.993%` · Mean IC: `0.0169`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.899 | 2.305 | 102.401% | 0.0248 |
| 2011 | 1.082 | 2.248 | 100.678% | 0.0168 |
| 2012 | 0.017 | 1.553 | 100.16% | 0.0243 |
| 2013 | -0.575 | 0.988 | 99.613% | 0.0161 |
| 2014 | 0.124 | 1.35 | 99.772% | 0.0183 |
| 2015 | 2.335 | 3.406 | 99.106% | 0.0245 |
| 2016 | -0.103 | 0.837 | 99.335% | 0.0132 |
| 2017 | -0.151 | 1.335 | 98.883% | 0.0153 |
| 2018 | -0.533 | 0.718 | 99.489% | 0.0105 |
| 2019 | 0.491 | 1.785 | 99.759% | 0.0201 |
| 2020 | 1.624 | 2.007 | 99.927% | 0.0146 |
| 2021 | 1.079 | 1.711 | 99.051% | 0.0128 |
| 2022 | -0.09 | 0.487 | 99.82% | 0.0116 |
| 2023 | 1.488 | 2.493 | 99.293% | 0.019 |
| 2024 | 1.572 | 2.454 | 98.505% | 0.0148 |
| 2025 | 1.743 | 2.018 | 98.646% | 0.0151 |
| 2026 | 2.021 | 2.695 | 96.648% | 0.0154 |

## Alpha 012


**Overall** — Net Sharpe: `-1.19` · Gross Sharpe: `-0.011` · Turnover: `121.675%` · Mean IC: `0.0084`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.718 | -0.996 | 123.294% | 0.0055 |
| 2011 | -2.329 | -0.96 | 121.859% | 0.0098 |
| 2012 | -0.259 | 1.675 | 122.502% | 0.0105 |
| 2013 | -0.649 | 1.447 | 122.256% | 0.01 |
| 2014 | -0.076 | 1.592 | 121.623% | 0.0105 |
| 2015 | -1.356 | 0.135 | 121.068% | 0.009 |
| 2016 | -0.528 | 0.639 | 122.224% | 0.0073 |
| 2017 | -1.331 | 0.561 | 122.429% | 0.0087 |
| 2018 | -2.044 | -0.268 | 122.296% | 0.0037 |
| 2019 | -0.567 | 1.28 | 121.564% | 0.0092 |
| 2020 | -1.753 | -1.081 | 120.103% | 0.0113 |
| 2021 | -0.688 | 0.112 | 119.628% | 0.0098 |
| 2022 | -0.867 | -0.031 | 120.755% | 0.0066 |
| 2023 | -2.34 | -1.024 | 120.143% | 0.0057 |
| 2024 | -1.421 | -0.396 | 120.227% | 0.01 |
| 2025 | -0.995 | -0.078 | 118.705% | 0.0068 |
| 2026 | -1.678 | -0.486 | 117.042% | 0.0068 |

## Alpha 013


**Overall** — Net Sharpe: `-0.635` · Gross Sharpe: `-0.072` · Turnover: `61.966%` · Mean IC: `0.0107`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.185 | 1.167 | 62.181% | 0.0114 |
| 2011 | 2.502 | 3.5 | 62.036% | 0.0134 |
| 2012 | 0.885 | 1.91 | 61.78% | 0.0141 |
| 2013 | 1.661 | 2.779 | 61.989% | 0.0106 |
| 2014 | 1.576 | 2.431 | 61.884% | 0.0131 |
| 2015 | -0.553 | 0.261 | 61.806% | 0.0093 |
| 2016 | -0.591 | 0.2 | 62.091% | 0.0089 |
| 2017 | -0.19 | 0.904 | 62.144% | 0.0109 |
| 2018 | 1.092 | 1.984 | 61.791% | 0.0086 |
| 2019 | -1.354 | -0.353 | 62.19% | 0.0093 |
| 2020 | -0.891 | -0.634 | 61.43% | 0.0127 |
| 2021 | -1.683 | -1.278 | 61.241% | 0.01 |
| 2022 | -1.338 | -0.92 | 61.362% | 0.004 |
| 2023 | -1.983 | -1.305 | 61.721% | 0.0079 |
| 2024 | -2.294 | -1.787 | 61.331% | 0.0121 |
| 2025 | -1.348 | -0.96 | 61.153% | 0.0155 |
| 2026 | -3.375 | -2.8 | 59.978% | 0.0112 |

## Alpha 014


**Overall** — Net Sharpe: `-0.51` · Gross Sharpe: `0.057` · Turnover: `68.445%` · Mean IC: `0.0035`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.174 | 1.101 | 70.35% | 0.0016 |
| 2011 | 0.053 | 0.708 | 70.153% | 0.0012 |
| 2012 | 1.012 | 2.219 | 68.683% | 0.0045 |
| 2013 | -1.571 | -0.234 | 69.914% | 0.0019 |
| 2014 | -0.259 | 0.621 | 70.034% | -0.0008 |
| 2015 | -0.841 | -0.045 | 69.099% | 0.0052 |
| 2016 | -1.289 | -0.556 | 69.76% | -0.0015 |
| 2017 | -0.543 | 0.54 | 69.302% | 0.0018 |
| 2018 | 0.098 | 1.033 | 68.141% | 0.0047 |
| 2019 | -1.223 | -0.277 | 67.648% | 0.0038 |
| 2020 | 0.006 | 0.358 | 68.465% | 0.0066 |
| 2021 | -1.16 | -0.764 | 67.625% | 0.0021 |
| 2022 | -0.009 | 0.306 | 66.066% | 0.0076 |
| 2023 | -1.619 | -0.916 | 65.742% | 0.0054 |
| 2024 | -2.311 | -1.712 | 65.168% | 0.0034 |
| 2025 | -0.029 | 0.306 | 66.227% | 0.0084 |
| 2026 | -1.088 | -0.635 | 64.179% | 0.0029 |

## Alpha 015


**Overall** — Net Sharpe: `-0.73` · Gross Sharpe: `-0.24` · Turnover: `61.624%` · Mean IC: `0.009`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 0.846 | 1.943 | 62.364% | 0.01 |
| 2011 | 2.341 | 3.476 | 61.728% | 0.0103 |
| 2012 | -0.124 | 1.067 | 61.492% | 0.0083 |
| 2013 | 0.546 | 1.748 | 61.399% | 0.0081 |
| 2014 | 0.203 | 1.287 | 61.357% | 0.0082 |
| 2015 | 0.37 | 1.36 | 61.546% | 0.0077 |
| 2016 | 0.533 | 1.548 | 61.735% | 0.0071 |
| 2017 | -1.408 | -0.198 | 61.543% | 0.0082 |
| 2018 | 0.214 | 1.335 | 61.692% | 0.0076 |
| 2019 | -1.417 | -0.306 | 61.451% | 0.0094 |
| 2020 | -0.339 | 0.009 | 61.036% | 0.0118 |
| 2021 | -1.919 | -1.48 | 60.939% | 0.0073 |
| 2022 | -1.764 | -1.212 | 61.252% | 0.0049 |
| 2023 | -0.916 | -0.157 | 61.219% | 0.0094 |
| 2024 | -2.946 | -2.373 | 61.097% | 0.0121 |
| 2025 | -1.726 | -1.564 | 60.804% | 0.0127 |
| 2026 | -1.634 | -1.095 | 60.056% | 0.011 |

## Alpha 016


**Overall** — Net Sharpe: `-0.515` · Gross Sharpe: `0.024` · Turnover: `62.985%` · Mean IC: `0.0113`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | 1.127 | 2.073 | 63.383% | 0.0101 |
| 2011 | 3.151 | 4.194 | 63.541% | 0.0139 |
| 2012 | 0.666 | 1.662 | 63.068% | 0.012 |
| 2013 | 0.47 | 1.553 | 62.911% | 0.0076 |
| 2014 | 1.429 | 2.268 | 62.836% | 0.0125 |
| 2015 | 0.191 | 1.083 | 62.927% | 0.0078 |
| 2016 | 0.386 | 1.212 | 63.324% | 0.0082 |
| 2017 | -1.44 | -0.374 | 63.205% | 0.0094 |
| 2018 | 0.667 | 1.617 | 63.215% | 0.0096 |
| 2019 | -1.437 | -0.491 | 63.284% | 0.0104 |
| 2020 | -0.761 | -0.502 | 61.761% | 0.0137 |
| 2021 | -1.55 | -1.159 | 61.834% | 0.0122 |
| 2022 | -1.259 | -0.827 | 62.638% | 0.006 |
| 2023 | -1.653 | -1.013 | 62.688% | 0.0119 |
| 2024 | -2.786 | -2.325 | 62.197% | 0.0155 |
| 2025 | -0.55 | -0.248 | 61.514% | 0.0191 |
| 2026 | -1.636 | -1.131 | 61.228% | 0.0134 |

## Alpha 017


**Overall** — Net Sharpe: `-1.796` · Gross Sharpe: `-0.246` · Turnover: `134.234%` · Mean IC: `0.0178`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.786 | -0.01 | 131.942% | 0.0223 |
| 2011 | -3.012 | -0.6 | 133.183% | 0.0216 |
| 2012 | -4.228 | -1.038 | 132.232% | 0.0132 |
| 2013 | -5.883 | -2.635 | 131.957% | 0.0139 |
| 2014 | -3.035 | -0.513 | 133.386% | 0.0147 |
| 2015 | -3.291 | -1.083 | 133.296% | 0.0179 |
| 2016 | -1.083 | 0.624 | 133.415% | 0.0259 |
| 2017 | -2.397 | 0.433 | 133.484% | 0.0242 |
| 2018 | -2.838 | -0.306 | 134.01% | 0.0196 |
| 2019 | -2.728 | -0.439 | 133.456% | 0.0116 |
| 2020 | -1.462 | -0.544 | 134.407% | 0.0228 |
| 2021 | -0.85 | 0.204 | 134.29% | 0.0193 |
| 2022 | -2.066 | -0.966 | 133.856% | 0.0061 |
| 2023 | -1.525 | 0.476 | 135.265% | 0.0152 |
| 2024 | -2.506 | -0.872 | 135.647% | 0.0194 |
| 2025 | 0.303 | 1.16 | 135.571% | 0.0212 |
| 2026 | -3.417 | -2.028 | 133.271% | 0.009 |

## Alpha 018


**Overall** — Net Sharpe: `-0.502` · Gross Sharpe: `0.704` · Turnover: `102.895%` · Mean IC: `0.0169`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -1.207 | 0.967 | 94.6% | 0.0174 |
| 2011 | -1.325 | 0.542 | 101.061% | 0.0187 |
| 2012 | -2.08 | 0.27 | 94.669% | 0.0183 |
| 2013 | -2.658 | -0.003 | 94.959% | 0.0142 |
| 2014 | -2.07 | -0.242 | 99.405% | 0.0168 |
| 2015 | -1.322 | 0.508 | 102.443% | 0.0239 |
| 2016 | -0.03 | 1.419 | 102.283% | 0.0271 |
| 2017 | -2.148 | 0.228 | 99.232% | 0.0171 |
| 2018 | -1.185 | 0.657 | 105.57% | 0.0186 |
| 2019 | -0.784 | 1.173 | 102.957% | 0.0122 |
| 2020 | -0.885 | -0.24 | 110.657% | 0.0173 |
| 2021 | 1.125 | 2.11 | 109.451% | 0.0244 |
| 2022 | -2.021 | -1.14 | 109.081% | 0.0101 |
| 2023 | 0.734 | 2.348 | 104.534% | 0.0123 |
| 2024 | -0.369 | 1.142 | 104.798% | 0.0097 |
| 2025 | 1.628 | 2.318 | 103.962% | 0.0147 |
| 2026 | 0.239 | 1.255 | 102.631% | 0.0098 |

## Alpha 019


**Overall** — Net Sharpe: `-0.166` · Gross Sharpe: `0.332` · Turnover: `48.478%` · Mean IC: `0.0181`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.814 | -0.492 | 69.022% | 0.0734 |
| 2011 | 0.878 | 1.646 | 51.972% | 0.0266 |
| 2012 | 0.273 | 1.465 | 50.599% | 0.0289 |
| 2013 | 0.046 | 1.104 | 48.007% | 0.0215 |
| 2014 | -0.7 | 0.109 | 47.203% | 0.0187 |
| 2015 | -0.567 | 0.246 | 52.229% | 0.0226 |
| 2016 | 0.441 | 1.105 | 48.077% | 0.0225 |
| 2017 | -0.147 | 0.898 | 46.94% | 0.017 |
| 2018 | -1.999 | -1.211 | 46.461% | 0.0113 |
| 2019 | 1.994 | 2.896 | 46.234% | 0.0211 |
| 2020 | 0.858 | 1.117 | 47.636% | 0.02 |
| 2021 | -0.444 | -0.104 | 49.214% | 0.0134 |
| 2022 | -2.21 | -1.834 | 48.71% | 0.0031 |
| 2023 | 1.082 | 1.842 | 47.0% | 0.0184 |
| 2024 | -0.522 | 0.118 | 46.442% | 0.0138 |
| 2025 | -0.41 | -0.086 | 46.814% | 0.0146 |
| 2026 | -0.64 | -0.257 | 46.486% | 0.0102 |

## Alpha 020


**Overall** — Net Sharpe: `-1.215` · Gross Sharpe: `0.382` · Turnover: `126.87%` · Mean IC: `0.0064`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.029 | 1.062 | 126.841% | 0.0064 |
| 2011 | -1.847 | 0.515 | 125.707% | 0.0084 |
| 2012 | -2.946 | 0.566 | 128.047% | -0.0031 |
| 2013 | -2.905 | 0.741 | 126.029% | 0.0025 |
| 2014 | -1.873 | 0.68 | 126.49% | 0.0078 |
| 2015 | -1.042 | 1.031 | 125.972% | 0.0074 |
| 2016 | -0.465 | 1.337 | 126.663% | 0.0065 |
| 2017 | -1.807 | 0.881 | 127.363% | 0.0123 |
| 2018 | -1.916 | 0.06 | 127.811% | 0.0045 |
| 2019 | -1.77 | 0.523 | 127.819% | 0.0075 |
| 2020 | 0.058 | 0.973 | 125.939% | 0.0157 |
| 2021 | -1.563 | -0.44 | 127.976% | 0.0006 |
| 2022 | -0.765 | 0.237 | 123.84% | 0.0018 |
| 2023 | -1.105 | 0.669 | 126.687% | 0.0082 |
| 2024 | -2.306 | -0.422 | 124.887% | 0.0068 |
| 2025 | -1.326 | -0.278 | 124.661% | 0.0079 |
| 2026 | -0.745 | 0.318 | 124.893% | 0.0109 |

## Alpha 021


**Overall** — Net Sharpe: `-1.677` · Gross Sharpe: `-0.41` · Turnover: `79.317%` · Mean IC: `-0.0114`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -2.1 | 0.285 | 84.111% | -0.0221 |
| 2011 | -2.712 | -0.667 | 85.269% | -0.0014 |
| 2012 | -2.596 | 0.551 | 81.784% | -0.0147 |
| 2013 | -4.46 | -1.256 | 79.231% | -0.0366 |
| 2014 | -3.877 | -1.087 | 78.548% | -0.0156 |
| 2015 | -3.342 | -1.083 | 77.758% | -0.0044 |
| 2016 | -2.641 | -0.838 | 79.296% | -0.0235 |
| 2017 | -3.54 | -0.656 | 75.225% | -0.0195 |
| 2018 | -2.202 | 0.21 | 78.477% | 0.0006 |
| 2019 | -3.55 | -0.971 | 78.553% | -0.0282 |
| 2020 | -0.167 | 0.656 | 79.603% | 0.002 |
| 2021 | -3.142 | -2.08 | 76.486% | -0.0135 |
| 2022 | -1.41 | -0.234 | 81.661% | 0.0079 |
| 2023 | -3.759 | -1.88 | 78.073% | -0.0071 |
| 2024 | -3.141 | -1.872 | 76.767% | -0.0035 |
| 2025 | 0.085 | 0.546 | 75.552% | -0.0052 |
| 2026 | -2.748 | -1.562 | 72.944% | -0.0044 |

## Alpha 022


**Overall** — Net Sharpe: `-0.991` · Gross Sharpe: `0.662` · Turnover: `72.976%` · Mean IC: `0.0039`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -1.664 | 0.971 | 73.308% | -0.0017 |
| 2011 | -1.551 | 0.89 | 72.799% | 0.0054 |
| 2012 | -1.352 | 1.29 | 72.921% | 0.0049 |
| 2013 | -0.585 | 2.326 | 73.395% | 0.0086 |
| 2014 | -1.407 | 1.157 | 72.608% | 0.0053 |
| 2015 | 0.225 | 2.384 | 72.748% | 0.0107 |
| 2016 | -2.016 | -0.252 | 73.012% | 0.0031 |
| 2017 | -1.82 | 0.908 | 73.0% | 0.0072 |
| 2018 | -2.39 | 0.073 | 72.34% | -0.0013 |
| 2019 | -3.466 | -0.907 | 72.792% | -0.0032 |
| 2020 | 1.32 | 2.166 | 72.411% | 0.0067 |
| 2021 | -0.88 | 0.179 | 71.85% | 0.0039 |
| 2022 | -1.644 | -0.544 | 72.394% | 0.0017 |
| 2023 | -1.994 | -0.047 | 72.673% | 0.006 |
| 2024 | -1.965 | -0.322 | 72.816% | 0.0013 |
| 2025 | -0.47 | 1.167 | 72.172% | 0.0048 |
| 2026 | 0.182 | 1.638 | 71.233% | 0.0036 |

## Alpha 023


**Overall** — Net Sharpe: `-0.552` · Gross Sharpe: `0.509` · Turnover: `91.862%` · Mean IC: `0.0033`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.39 | 1.529 | 95.097% | -0.0009 |
| 2011 | 0.52 | 1.945 | 96.991% | 0.0151 |
| 2012 | -0.469 | 1.927 | 93.05% | 0.0007 |
| 2013 | -1.371 | 1.118 | 89.368% | -0.0088 |
| 2014 | -1.311 | 0.754 | 90.814% | -0.0011 |
| 2015 | -0.541 | 1.072 | 93.13% | 0.0127 |
| 2016 | -0.022 | 1.307 | 91.917% | 0.0015 |
| 2017 | -2.27 | -0.003 | 87.145% | -0.001 |
| 2018 | -0.305 | 1.041 | 90.518% | 0.0051 |
| 2019 | -0.191 | 1.533 | 89.7% | -0.0028 |
| 2020 | -0.005 | 0.434 | 92.146% | 0.0029 |
| 2021 | -1.954 | -1.131 | 88.651% | 0.0009 |
| 2022 | -1.019 | -0.183 | 94.456% | 0.0075 |
| 2023 | 0.08 | 1.371 | 91.495% | 0.0061 |
| 2024 | -2.476 | -1.237 | 90.416% | 0.0056 |
| 2025 | -0.528 | 0.38 | 90.993% | 0.0056 |
| 2026 | 1.153 | 1.964 | 86.929% | 0.0129 |

## Alpha 024


**Overall** — Net Sharpe: `0.776` · Gross Sharpe: `1.312` · Turnover: `38.492%` · Mean IC: `0.0179`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.639 | 0.647 | 61.983% | 0.0263 |
| 2011 | -0.022 | 0.821 | 42.669% | 0.0234 |
| 2012 | 0.594 | 1.411 | 32.367% | 0.0249 |
| 2013 | 0.589 | 1.763 | 46.029% | 0.0227 |
| 2014 | -0.897 | -0.089 | 38.426% | 0.0145 |
| 2015 | 0.637 | 1.28 | 35.109% | 0.0266 |
| 2016 | 0.576 | 1.129 | 31.139% | 0.0178 |
| 2017 | 0.098 | 0.947 | 38.482% | 0.0174 |
| 2018 | 0.633 | 1.423 | 39.9% | 0.0168 |
| 2019 | 0.508 | 1.129 | 29.416% | 0.0111 |
| 2020 | 2.37 | 2.661 | 32.045% | 0.0164 |
| 2021 | -0.173 | 0.259 | 50.666% | 0.0208 |
| 2022 | 0.384 | 0.687 | 29.019% | 0.014 |
| 2023 | 0.928 | 1.481 | 32.833% | 0.0092 |
| 2024 | 1.899 | 2.646 | 37.937% | 0.0171 |
| 2025 | 2.321 | 2.628 | 34.832% | 0.0091 |
| 2026 | 2.25 | 2.611 | 42.33% | 0.015 |

## Alpha 025


**Overall** — Net Sharpe: `-1.125` · Gross Sharpe: `0.273` · Turnover: `124.541%` · Mean IC: `0.0152`

| Year | Net Sharpe | Gross Sharpe | Turnover | Mean IC |
|------|------------|--------------|----------|---------|
| 2010 | -0.843 | 1.593 | 126.161% | 0.0158 |
| 2011 | -2.032 | -0.03 | 125.792% | 0.0096 |
| 2012 | -3.185 | -0.599 | 123.964% | 0.0038 |
| 2013 | -4.4 | -1.469 | 124.642% | 0.0076 |
| 2014 | -2.607 | -0.66 | 124.566% | 0.0089 |
| 2015 | -1.019 | 0.962 | 125.541% | 0.0225 |
| 2016 | 0.158 | 1.73 | 126.816% | 0.0288 |
| 2017 | -1.702 | 0.823 | 124.567% | 0.0219 |
| 2018 | -0.964 | 1.253 | 122.988% | 0.0153 |
| 2019 | -1.688 | 0.361 | 123.223% | 0.0096 |
| 2020 | -1.101 | -0.309 | 124.619% | 0.0204 |
| 2021 | 0.164 | 1.102 | 123.041% | 0.0243 |
| 2022 | -2.276 | -1.403 | 122.474% | 0.0071 |
| 2023 | 0.187 | 1.933 | 122.872% | 0.0165 |
| 2024 | -2.378 | -0.788 | 121.944% | 0.0136 |
| 2025 | 0.042 | 0.967 | 122.741% | 0.0171 |
| 2026 | -0.563 | 0.509 | 120.298% | 0.0153 |

